# Data acquisition

This chapter pulls TTWA boundaries from the ONS Open Geography Portal and workplace-to-residence commuting flows from NOMIS. Outputs are cached as `.rds` files so later chapters can run without repeating slow API calls.

**Previous:** [Introduction](01_introduction.ipynb)  
**Next:** [Data cleaning](03_data_cleaning.ipynb)


In [1]:
suppressPackageStartupMessages({
  library(tidyverse)
  library(sf)
  library(here)
})


In [2]:
proj_dir <- here::here("projects", "uk-urban-systems-network")
data_dir <- file.path(proj_dir, "data")
fig_dir <- file.path(proj_dir, "figures")
dir.create(data_dir, recursive = TRUE, showWarnings = FALSE)
dir.create(fig_dir, recursive = TRUE, showWarnings = FALSE)


## TTWA boundaries (ONS)

In [3]:
rds_ttwa <- file.path(data_dir, "ttwa_boundaries.rds")
rds_od <- file.path(data_dir, "od_flows_raw.rds")
rds_lookup <- file.path(data_dir, "msoa_ttwa_lookup.rds")

if (!all(file.exists(c(rds_ttwa, rds_od, rds_lookup)))) {
  message("Running live data fetch (ONS boundaries + NOMIS ODWP01EW bulk CSV) …")
  fetch_script <- file.path(proj_dir, "scripts", "fetch_live_data.R")
  if (!file.exists(fetch_script)) {
    stop("Missing fetch script: ", fetch_script)
  }
  status <- system2("Rscript", c(fetch_script), stdout = TRUE, stderr = TRUE)
  if (!is.null(attr(status, "status")) && attr(status, "status") != 0) {
    stop("fetch_live_data.R failed:\n", paste(status, collapse = "\n"))
  }
}

ttwa_boundaries <- readRDS(rds_ttwa)
message("Loaded TTWA boundaries: ", nrow(ttwa_boundaries), " areas")


Loaded TTWA boundaries: 228 areas



## Commuting flows (NOMIS)

In [4]:
# 2021 Census ODWP01EW (bulk download; not available via nomisr API)
od_flows_raw <- readRDS(rds_od)
message("Loaded OD flows: ", nrow(od_flows_raw), " MSOA pairs")


Loaded OD flows: 1828307 MSOA pairs



## Preview and summary

In [5]:
print(head(od_flows_raw))

print(od_flows_raw |> count(source, geo_level))

country_lookup <- tibble(
  country = c("E", "W", "S", "N"),
  country_name = c("England", "Wales", "Scotland", "Northern Ireland")
)

ttwa_country_summary <- ttwa_boundaries |>
  st_drop_geometry() |>
  count(country, name = "n_ttwa") |>
  left_join(country_lookup, by = "country") |>
  select(country_name, n_ttwa)

print(ttwa_country_summary)


  residence_msoa workplace_msoa flow
1      E02000001      E02000001  436
2      E02000001      E02000016    2
3      E02000001      E02000024    3
4      E02000001      E02000027    1
5      E02000001      E02000055    1
6      E02000001      E02000060    1


      country_name n_ttwa
1          England    155
2 Northern Ireland     10
3         Scotland     45
4            Wales     18


## Data sources

[^ons]: ONS Open Geography Portal, TTWA (December 2011) UK BGC boundaries.
[^nomis]: NOMIS, 2021 Census table ODWP01EW (workplace–residence flows).

---

**Next:** [Data cleaning →](03_data_cleaning.ipynb)
